# Multivariate Generation Forecasting

## Importing Forecast Data 

- Plant
- Weather 
- availability
- curtailment 
- irradiance
- health factor

In [1]:
import pandas as pd
import os

from src.irradiance import predict_irradiance
from src.generation_univariate import predict_generation
from src.availability import predict_availability
from src.health_factor import predict_health_factor
from src.curtailment import predict_curtailment
#from src.multivariate import predict_multivariate

In [2]:


plant = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\plant_master.csv")
weather = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\weather_forecast.csv", parse_dates=["timestamp"])

In [3]:
info = plant.merge(weather, on='plant_name', how='left')
plant_id = 'PLANT_001'
info = info[info['plant_id'] == plant_id]
info.shape

(72, 17)

In [4]:

# Define the base folder path for irradiance models
irradiance_folder = r"D:\Portfolio Github\Pravaah\models\irradiance"
generation_folder = r"D:\Portfolio Github\Pravaah\models\generation"
availability_folder = r"D:\Portfolio Github\Pravaah\models\availability"
health_factor_folder = r"D:\Portfolio Github\Pravaah\models\health_factor"
curtailment_folder = r"D:\Portfolio Github\Pravaah\models\curtailment"


# List all files and filter for the plant_id
irradiance_files = [f for f in os.listdir(irradiance_folder) if plant_id in f and f.endswith('.pkl')]
generation_files = [f for f in os.listdir(generation_folder) if plant_id in f and f.endswith('.pkl')]
availability_files = [f for f in os.listdir(availability_folder) if plant_id in f and f.endswith('.pkl')]
health_factor_files = [f for f in os.listdir(health_factor_folder) if plant_id in f and f.endswith('.pkl')]
curtailment_files = [f for f in os.listdir(curtailment_folder) if plant_id in f and f.endswith('.pkl')]


if irradiance_files:
    model_path = os.path.join(irradiance_folder, irradiance_files[0])
    print(f"Found model: {model_path}")
    irradiance = predict_irradiance(model_path, info['timestamp'])
else:
    print(f"No model found for {plant_id} in {irradiance_folder}")

if health_factor_files:
    model_path = os.path.join(health_factor_folder, health_factor_files[0])
    print(f"Found model: {model_path}")
    health_factor = predict_health_factor(model_path, info['timestamp'])
else:
    print(f"No model found for {plant_id} in {health_factor_folder}")

if availability_files:
    model_path = os.path.join(availability_folder, availability_files[0])
    print(f"Found model: {model_path}")
    availability = predict_availability(model_path, info['timestamp'])
else:
    print(f"No model found for {plant_id} in {availability_folder}")

if generation_files:
    model_path = os.path.join(generation_folder, generation_files[0])
    print(f"Found model: {model_path}")
    #info['timestamp'] = info['timestamp'].sort_values().reset_index(drop=True)
    #info['timestamp'] = pd.to_datetime(info['timestamp'])
    generation = predict_generation(model_path, info['timestamp'])
else:
    print(f"No model found for {plant_id} in {generation_folder}")

merged_df = irradiance[['plant_id', 'timestamp', 'irradiance_forecast']].merge(health_factor[['timestamp', 'health_forecast']], on='timestamp', how='left')
merged_df = merged_df.merge(generation[['timestamp', 'forecast_mw']], on='timestamp', how='left')
merged_df = merged_df.merge(availability[['timestamp', 'availability_forecast']], on='timestamp', how='left')
merged_df = merged_df.merge(info[['timestamp','plant_type', 'capacity_mw']], on='timestamp', how='left')


if curtailment_files:
    model_path = os.path.join(curtailment_folder, curtailment_files[0])
    print(f"Found model: {model_path}")
    curtailment = predict_curtailment(model_path,merged_df[[ 'timestamp','irradiance_forecast','forecast_mw']])
else:
    print(f"No model found for {plant_id} in {curtailment_folder}")

merged_df = merged_df.merge(curtailment[['timestamp', 'curtailment_mw_forecast']], on=['timestamp'], how='left') 
merged_df = merged_df.merge(weather, on=['timestamp'], how='left')
merged_df.head()

Found model: D:\Portfolio Github\Pravaah\models\irradiance\PLANT_001_irradiance_20260505_1348.pkl
Found model: D:\Portfolio Github\Pravaah\models\health_factor\PLANT_001_health_20260505_1808.pkl
Found model: D:\Portfolio Github\Pravaah\models\availability\PLANT_001_availability_20260505_1939.pkl
Found model: D:\Portfolio Github\Pravaah\models\generation\PLANT_001_generation_model_20260506_0656.pkl
Found model: D:\Portfolio Github\Pravaah\models\curtailment\PLANT_001_curtailment_20260506_1821.pkl


,plant_id,timestamp,irradiance_forecast,health_forecast,forecast_mw,availability_forecast,plant_type,capacity_mw,curtailment_mw_forecast,temperature,cloud_cover,wind_speed,wind_direction,irradiance,plant_name
0,PLANT_001,2025-04-24,0.0,0.7814,NaN,86.56,Solar,102,0.072,29.3,0,17.2,170,0.0,Koppal Solar
1,PLANT_001,2025-04-24,0.0,0.7814,NaN,86.56,Solar,102,0.072,27.2,1,10.6,178,0.0,Raichur Wind
2,PLANT_001,2025-04-24,0.0,0.7814,NaN,86.56,Solar,102,0.072,28.9,0,11.1,228,0.0,Gadag Renewables
3,PLANT_001,2025-04-24,0.0,0.7814,NaN,86.56,Solar,102,0.072,25.1,91,17.7,260,0.0,Bidar Solar
4,PLANT_001,2025-04-24,0.0,0.7814,NaN,86.56,Solar,102,0.072,27.7,0,9.4,179,0.0,Ballari Wind


In [5]:
merged_df.rename(columns={
    "curtailment_mw_forecast": "curtailment_mw",
    'availability_forecast': "availability_mw",
    "irradiance_forecast": "irradiance_wm2",
    "health_forecast": "health_factor", 
    "forecast_mw": "generation"
}, inplace=True)
merged_df.columns
#merged_df.drop(["plant_name", "latitude", "longitude", "commission_date", "region", "state", "developer"], axis=1, inplace=True)


Index(['plant_id', 'timestamp', 'irradiance_wm2', 'health_factor',
       'generation', 'availability_mw', 'plant_type', 'capacity_mw',
       'curtailment_mw', 'temperature', 'cloud_cover', 'wind_speed',
       'wind_direction', 'irradiance', 'plant_name'],
      dtype='str')

In [21]:
merged_df.head()

,plant_id,plant_name,plant_type,capacity_mw,latitude,longitude,commission_date,region,state,developer,...,actual_generation_mw,availability_mw,curtailment_mw,health_factor,status,temperature,cloud_cover,wind_speed,wind_direction,irradiance
0,PLANT_001,Koppal Solar,Solar,102,17.2093,79.8116,2015-11-17,Kalyana Karnataka,Karnataka,ReNew Power,...,0.0,87.39,0.00,0.857,ON,20.9,2.0,10.5,158.0,0.0
1,PLANT_001,Koppal Solar,Solar,102,17.2093,79.8116,2015-11-17,Kalyana Karnataka,Karnataka,ReNew Power,...,0.0,87.39,0.00,0.857,ON,20.3,11.0,8.4,155.0,0.0
2,PLANT_001,Koppal Solar,Solar,102,17.2093,79.8116,2015-11-17,Kalyana Karnataka,Karnataka,ReNew Power,...,0.0,87.39,2.45,0.857,ON,19.8,22.0,5.5,148.0,0.0
3,PLANT_001,Koppal Solar,Solar,102,17.2093,79.8116,2015-11-17,Kalyana Karnataka,Karnataka,ReNew Power,...,0.0,87.39,0.00,0.857,ON,19.3,31.0,6.4,133.0,0.0
4,PLANT_001,Koppal Solar,Solar,102,17.2093,79.8116,2015-11-17,Kalyana Karnataka,Karnataka,ReNew Power,...,0.0,87.39,0.00,0.857,ON,19.1,1.0,6.5,124.0,0.0


## Importing Historical Data 

- Plant
- Generation
- Weather

In [6]:
import pandas as pd

plant = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\plant_master.csv")
weather_historical = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\weather_historical.csv", parse_dates=["timestamp"])
generation_historical = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\generation.csv", parse_dates=["timestamp"])


# Merge plant with generation on 'plant_id'
merged_df = plant.merge(generation_historical, on=['plant_id', 'plant_type' ,'region'], how='left')

# Merge with weather on 'plant_id' and 'timestamp'
merged_df = merged_df.merge(weather_historical, on=['plant_name', 'timestamp'], how='left')

historical_df = merged_df.drop(columns=[ 'plant_name', 
    'latitude', 'longitude', 'commission_date', 'region', 'state',
    'developer', 'offtaker','status',
])

historical_df.columns

Index(['plant_id', 'plant_type', 'capacity_mw', 'timestamp',
       'actual_generation_mw', 'availability_mw', 'curtailment_mw',
       'health_factor', 'irradiance_wm2', 'temperature', 'cloud_cover',
       'wind_speed', 'wind_direction', 'irradiance'],
      dtype='str')

## Pre-processing

In [7]:
from src.preprocessing import preprocess

historical_df = preprocess(historical_df)
forecast_df = preprocess(merged_df,is_forecast=True)


────────────────────────────────────────────────────────────
  STARTING PRE-PROCESSING PIPELINE
────────────────────────────────────────────────────────────
[INFO] irradiance_wm2 vs irradiance correlation: 0.6111
[OK] Merged irradiance_wm2 + irradiance → single 'irradiance' column
[OK] Numeric dtypes enforced
[OK] Imputation complete — nulls reduced: 919,880 → 229,940
[WARN] Remaining nulls:
temperature    229940
dtype: int64
[OK] Outlier treatment done (iqr) on: ['actual_generation_mw', 'availability_mw', 'temperature', 'cloud_cover', 'wind_speed', 'irradiance']
[OK] Dtypes optimised — memory usage: 47.1 MB

════════════════════════════════════════════════════════════
  PRE-PROCESSING QA REPORT
════════════════════════════════════════════════════════════
  Shape          : 574,850 rows × 14 columns
  Plants         : 50
  Plant types    : {'Solar': 287425, 'Wind': 241437, 'Hybrid': 45988}
  Date range     : 2024-01-01 00:00:00 → 2025-04-24 00:00:00
  Null values    : 229940
  Duplica

In [8]:
historical_df.columns

Index(['plant_id', 'plant_type', 'capacity_mw', 'timestamp',
       'actual_generation_mw', 'availability_mw', 'curtailment_mw',
       'health_factor', 'temperature', 'cloud_cover', 'wind_speed',
       'wind_direction', 'irradiance', 'generation'],
      dtype='str')

In [9]:
forecast_df.columns

Index(['plant_id', 'plant_name', 'plant_type', 'capacity_mw', 'latitude',
       'longitude', 'commission_date', 'region', 'state', 'developer',
       'offtaker', 'timestamp', 'actual_generation_mw', 'availability_mw',
       'curtailment_mw', 'health_factor', 'status', 'temperature',
       'cloud_cover', 'wind_speed', 'wind_direction', 'irradiance'],
      dtype='str')

## Derived Features

In [10]:

from src.features import build_features

historical_df = build_features(historical_df, "Solar")
historical_df.drop(columns=['plant_type'], inplace=True)


forecast_df = build_features(forecast_df, "Solar")
forecast_df.drop(columns=['plant_type'], inplace=True)

In [11]:
historical_df.rename(columns={
    'actual_generation_mw': 'generation'}, inplace=True)

In [12]:
forecast_df.rename(columns={
    'actual_generation_mw': 'generation'}, inplace=True)

In [13]:
historical_df = historical_df.loc[:, ~historical_df.columns.duplicated()]
forecast_df   = forecast_df.loc[:, ~forecast_df.columns.duplicated()]

In [19]:
historical_df =    historical_df[historical_df['plant_id']== 'PLANT_001']
forecast_df =    forecast_df[forecast_df['plant_id']== 'PLANT_001']

## Forecasting

In [14]:
from src.multivariate import (
    run_multivariate_fleet,   # full fleet: train + forecast + simulate
    run_single_plant,         # one plant: debug / inspect
    run_decomposition_only,   # STL decomposition fleet-wide (no training)
    decompose_series,         # STL for one plant
    simulate_scenarios,       # Monte Carlo standalone
    select_best_multivariate_model,  # train + forecast one plant (returns dict)
    merge_for_multivariate,   # stack historical + forecast rows
)


In [20]:
all_forecasts_df, all_simulations_df = run_multivariate_fleet(
    historical_df = historical_df,
    forecast_df   = forecast_df,
    feature_cols  = None,      # None → auto-select from EXOGENOUS + ENDOGENOUS
    val_days      = 14,        # walk-forward fold size in days
    n_splits      = 3,         # number of walk-forward folds
    n_simulations = 1000,      # Monte Carlo paths per plant
    n_jobs        = -1,        # -1 = use all CPU cores
    output_dir    = "data/multivariate/",
)


Merging DataFrames...
[Merge] historical=11,497  forecast=11,497  plants=1
[Merge] Extended: 11,497 historical + 11,497 forecast rows

════════════════════════════════════════════════════════════
  MULTIVARIATE FLEET RUN
  Plants      : 1
  Target      : generation  (MW)
  Horizon     : t+1 .. t+72h
  Val window  : 14 days × 3 folds
  MC paths    : 1000 / plant
  Workers     : 1 / 8 cores
════════════════════════════════════════════════════════════


════════════════════════════════════════════════════════════
  MULTIVARIATE: PLANT_001
════════════════════════════════════════════════════════════
  [PLANT_001] Dropped 84 training rows with NaN features
[PLANT_001] train=11,413  future=11,497  features=41
[PLANT_001] STL:
  daily seasonal : 0.967  (strong)
  weekly seasonal: 0.966  (strong)
  trend          : 0.021
  residual std   : 3.475 MW
  dominant       : daily  |  difficulty: easy

  Walk-forward CV (3 folds × 14 days):

  Fold 1/3: train=11,077  val=336
    lightgbm     → MAE=0.

[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:  1.7min finished


  Saved to data\multivariate/


In [22]:
stl_summary = run_decomposition_only(
    historical_df=historical_df,
    output_csv="data/multivariate/stl_fleet_summary.csv",
)

stl_summary

[PLANT_001] STL:
  daily seasonal : 0.968  (strong)
  weekly seasonal: 0.965  (strong)
  trend          : 0.023
  residual std   : 3.465 MW
  dominant       : daily  |  difficulty: easy

[Decompose] Fleet STL — 1 plants
  easy  : 1 plants (100%)
  medium: 0 plants (0%)
  hard  : 0 plants (0%)
  Saved: data/multivariate/stl_fleet_summary.csv


,plant_id,seasonal_strength,weekly_strength,trend_strength,residual_std,dominant_period,forecast_difficulty
0,PLANT_001,0.9677,0.9655,0.023,3.4655,daily,easy


In [23]:

# Inspect difficulty distribution
print("\nFleet difficulty breakdown:")
print(stl_summary["forecast_difficulty"].value_counts())

# Focus on hard plants that need more attention
hard_plants = stl_summary[stl_summary["forecast_difficulty"] == "hard"]
print(f"\nHard plants ({len(hard_plants)}):")
print(hard_plants[["plant_id", "residual_std", "seasonal_strength"]].to_string(index=False))


Fleet difficulty breakdown:
forecast_difficulty
easy    1
Name: count, dtype: int64

Hard plants (0):
Empty DataFrame
Columns: [plant_id, residual_std, seasonal_strength]
Index: []
